## Step 1: Read Bronze Table

In [0]:
from pyspark.sql.functions import *
df_bronze = spark.table(
    "nyc_taxi.bronze.taxi_trip_bronze"
    )
print(f"Bronze count: {df_bronze.count()}")
display(df_bronze.limit(10))

Bronze count: 11077206


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,ingestion_timestamp,source_file_path,source_file_name
7,2026-02-01T00:05:57.000,2026-02-01T00:05:57.000,1,0.94,1,N,107,170,1,7.2,0.0,0.5,0.0,0.0,1.0,12.95,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
7,2026-02-01T00:35:58.000,2026-02-01T00:35:58.000,1,1.93,1,N,234,141,1,11.4,0.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
2,2026-02-01T00:08:41.000,2026-02-01T00:39:32.000,1,9.99,1,N,138,68,1,44.3,6.0,0.5,11.01,0.0,1.0,67.81,2.5,1.75,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
1,2026-02-01T00:29:06.000,2026-02-01T00:41:04.000,0,1.7,1,N,209,13,1,12.8,4.25,0.5,3.7,0.0,1.0,22.25,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
1,2026-02-01T00:53:52.000,2026-02-01T01:11:21.000,0,3.7,1,N,249,229,1,19.8,4.25,0.5,6.35,0.0,1.0,31.9,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
2,2026-02-01T00:24:29.000,2026-02-01T00:36:01.000,1,1.6,1,N,113,90,1,12.1,1.0,0.5,0.09,0.0,1.0,17.94,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
2,2026-02-01T00:40:20.000,2026-02-01T00:54:57.000,2,1.73,1,N,234,144,1,14.2,1.0,0.5,1.0,0.0,1.0,20.95,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
2,2026-02-01T00:11:48.000,2026-02-01T00:22:41.000,1,2.27,1,N,237,151,1,12.8,1.0,0.5,2.0,0.0,1.0,19.8,2.5,0.0,0.0,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
2,2026-02-01T00:02:38.000,2026-02-01T00:26:38.000,1,4.92,1,N,148,263,1,26.1,1.0,0.5,4.78,0.0,1.0,36.63,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet
2,2026-02-01T00:05:56.000,2026-02-01T00:22:06.000,1,1.98,1,N,79,170,1,15.6,1.0,0.5,4.27,0.0,1.0,25.62,2.5,0.0,0.75,2026-06-27T19:37:59.506Z,dbfs:/Volumes/nyc_taxi/raw/taxi_files/yellow_tripdata_2026-02.parquet,yellow_tripdata_2026-02.parquet


## Step 2: Create Derived Columns

In [0]:
df_silver = (df_bronze\
            .withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))\
            .withColumn("pickup_hour",hour(col("tpep_pickup_datetime")))\
            .withColumn("trip_duration_minutes", (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime")))/60)\
            .withColumn("trip_year",year(col("tpep_pickup_datetime")))\
            .withColumn("trip_month",month(col("tpep_pickup_datetime")))
)

## Step 3: Quick Validation

In [0]:
df_silver.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes",
    "pickup_date",
    "pickup_hour",
    "trip_year",
    "trip_month"
    ).show(5, False)

+--------------------+---------------------+---------------------+-----------+-----------+---------+----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|pickup_date|pickup_hour|trip_year|trip_month|
+--------------------+---------------------+---------------------+-----------+-----------+---------+----------+
|2026-02-01 00:05:57 |2026-02-01 00:05:57  |0.0                  |2026-02-01 |0          |2026     |2         |
|2026-02-01 00:35:58 |2026-02-01 00:35:58  |0.0                  |2026-02-01 |0          |2026     |2         |
|2026-02-01 00:08:41 |2026-02-01 00:39:32  |30.85                |2026-02-01 |0          |2026     |2         |
|2026-02-01 00:29:06 |2026-02-01 00:41:04  |11.966666666666667   |2026-02-01 |0          |2026     |2         |
|2026-02-01 00:53:52 |2026-02-01 01:11:21  |17.483333333333334   |2026-02-01 |0          |2026     |2         |
+--------------------+---------------------+---------------------+-----------+-----------+---------+----

## Step 4: Data Quality Rules

In [0]:
from pyspark.sql.functions import col
valid_df = df_silver.filter(
    (col("passenger_count")>0) &
    (col("trip_distance")>0) &
    (col("fare_amount")>0) &
    (col("trip_duration_minutes")>0)
)

## Handling Null also in rejected records

In [0]:
rejected_df = df_silver.filter(
    (col("passenger_count").isNull()) |
    (col("trip_distance").isNull()) |
    (col("fare_amount").isNull()) |
    (col("trip_duration_minutes").isNull()) |
    (col("passenger_count")<=0) |
    (col("trip_distance")<=0) |
    (col("fare_amount")<=0) |
    (col("trip_duration_minutes")<=0)
)

## Step 5: Capture Metrics

In [0]:
df_silver.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file_path:

In [0]:
df_silver.filter(
    col("trip_duration_minutes") > 0
).count()

10942405

In [0]:
import builtins
total_records = df_silver.count()

valid_records = valid_df.count()

rejected_records = rejected_df.count()

df_percentage = builtins.round((valid_records/total_records)*100,2)

print(f"Total records: {total_records}")
print(f"Valid records: {valid_records}")
print(f"Rejected records: {rejected_records}")
print(f"Data Quality Percentage : {df_percentage}%")

print(f"Missing Records: {total_records-valid_records-rejected_records}")

Total records: 11077206
Valid records: 7668647
Rejected records: 3408559
Data Quality Percentage : 69.23%
Missing Records: 0


## Step 6: Trip Category

In [0]:
valid_df = (
    valid_df
    .withColumn("trip_category",
    when(col("trip_distance")<2 , "Short Trip")
    .when(col("trip_distance")<10,"Medium Trip")
    .otherwise("Long Trip")
    )
)

## Step 7: Deduplicate

In [0]:
before_count = valid_df.count()

valid_df = valid_df.dropDuplicates()

after_count = valid_df.count()

print(f"Before dropping duplicates: {before_count}")
print(f"After dropping duplicates: {after_count}")

Before dropping duplicates: 7668647
After dropping duplicates: 7668647


In [0]:
from pyspark.sql.functions import *

df_silver.select(
    sum(when(col("passenger_count").isNull(),1).otherwise(0)).alias("passenger_nulls"),
    sum(when(col("trip_distance").isNull(),1).otherwise(0)).alias("distance_nulls"),
    sum(when(col("fare_amount").isNull(),1).otherwise(0)).alias("fare_nulls"),
    sum(when(col("trip_duration_minutes").isNull(),1).otherwise(0)).alias("duration_nulls")
).show()

+---------------+--------------+----------+--------------+
|passenger_nulls|distance_nulls|fare_nulls|duration_nulls|
+---------------+--------------+----------+--------------+
|        3057123|             0|         0|             0|
+---------------+--------------+----------+--------------+



In [0]:
rejected_df.select(
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes"
).show(20, False)

+---------------+-------------+-----------+---------------------+
|passenger_count|trip_distance|fare_amount|trip_duration_minutes|
+---------------+-------------+-----------+---------------------+
|1              |0.0          |15.6       |0.0                  |
|1              |0.0          |7.9        |0.0                  |
|2              |0.0          |28.35      |0.13333333333333333  |
|1              |0.0          |30.3       |0.0                  |
|1              |0.0          |10.7       |0.16666666666666666  |
|1              |0.0          |30.0       |0.06666666666666667  |
|1              |0.0          |3.0        |0.55                 |
|1              |3.12         |20.5       |0.0                  |
|1              |1.34         |12.1       |0.0                  |
|1              |0.0          |-3.0       |0.6833333333333333   |
|1              |0.0          |3.0        |0.6833333333333333   |
|4              |0.0          |15.0       |0.25                 |
|4        

In [0]:
(
    valid_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.silver.taxi_trip_silver")
)

In [0]:
(
    rejected_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.silver.taxi_trip_rejected")
)

In [0]:
from pyspark.sql import Row
from datetime import datetime

metrics = [
    Row(
        total_records=total_records,
        valid_records=valid_records,
        rejected_records=rejected_records,
        dq_percentage=df_percentage,
        load_timestamp=datetime.now()
    )
]

dq_metrics_df = spark.createDataFrame(metrics)

In [0]:
(
    dq_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi.silver.dq_metrics")
)

In [0]:
%sql
select count(*) from nyc_taxi.silver.taxi_trip_silver

count(*)
7668647


In [0]:
%sql
select count(*) from nyc_taxi.silver.taxi_trip_rejected

count(*)
3408559


In [0]:
%sql
select * from nyc_taxi.silver.dq_metrics

total_records,valid_records,rejected_records,dq_percentage,load_timestamp
11077206,7668647,3408559,69.23,2026-06-28T06:40:03.850Z
